# Experiment 5: Create Container for the FastAPI ML Service (Docker)

**Objective:**
- Create a Dockerfile for the FastAPI ML service
- Create a requirements.txt
- Build and run the Docker container
- Test the containerized service

**Prerequisites:** Run Experiments 1-4 first. Docker must be installed.

## Step 1: Install Docker (Instructions)

### macOS:
```bash
brew install --cask docker
# OR download from https://docs.docker.com/desktop/install/mac-install/
```

### Windows (WSL2 required):
1. Enable WSL2: `wsl --install`
2. Download Docker Desktop from https://docs.docker.com/desktop/install/windows-install/
3. Enable WSL2 backend in Docker Desktop settings

### Linux:
```bash
sudo apt-get update
sudo apt-get install docker.io docker-compose
sudo systemctl start docker
sudo usermod -aG docker $USER
```

## Step 2: Verify Docker Installation

In [ ]:
import subprocess
import os

# Check Docker version
try:
    result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
    print(f"Docker Version: {result.stdout.strip()}")
except FileNotFoundError:
    print("Docker is not installed. Please install Docker first.")

# Check Docker Compose
try:
    result = subprocess.run(['docker', 'compose', 'version'], capture_output=True, text=True)
    print(f"Docker Compose: {result.stdout.strip()}")
except:
    print("Docker Compose not available")

## Step 3: Create requirements.txt

In [ ]:
requirements = """fastapi==0.115.0
uvicorn[standard]==0.30.0
pydantic==2.9.0
scikit-learn==1.5.0
pandas==2.2.0
numpy==1.26.0
python-jose[cryptography]==3.3.0
passlib[bcrypt]==1.7.4
python-multipart==0.0.9
python-json-logger==2.0.7
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt created:")
print(requirements)

## Step 4: Create the Dockerfile

In [ ]:
dockerfile_content = """# Use Python 3.11 slim image as base
FROM python:3.11-slim

# Set metadata
LABEL maintainer="MLOps Course"
LABEL description="Bank Churn Prediction FastAPI Service"
LABEL version="1.0"

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV SECRET_KEY=mlops-secret-key-2024
ENV API_KEYS=mlops-api-key-001,mlops-api-key-002

# Set working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (for Docker cache optimization)
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir --upgrade pip && \\
    pip install --no-cache-dir -r requirements.txt

# Copy model artifacts
COPY model_artifacts/ ./model_artifacts/

# Copy application code
COPY app.py .

# Create logs directory
RUN mkdir -p logs

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1

# Run the application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print("Dockerfile created:")
print(dockerfile_content)

## Step 5: Create .dockerignore

In [ ]:
dockerignore_content = """__pycache__
*.pyc
*.pyo
.git
.gitignore
.env
*.ipynb
*.csv
logs/
.vscode/
.idea/
*.md
venv/
env/
.pytest_cache/
node_modules/
frontend/
"""

with open('.dockerignore', 'w') as f:
    f.write(dockerignore_content)

print(".dockerignore created!")

## Step 6: Create docker-compose.yml

In [ ]:
docker_compose_content = """version: '3.8'

services:
  churn-prediction-api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: churn-prediction-api
    ports:
      - "8000:8000"
    environment:
      - SECRET_KEY=mlops-secret-key-2024
      - API_KEYS=mlops-api-key-001,mlops-api-key-002
    volumes:
      - ./logs:/app/logs
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 10s
"""

with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose_content)

print("docker-compose.yml created!")

## Step 7: Build the Docker Image

In [ ]:
# Build the Docker image
print("Building Docker image...")
result = subprocess.run(
    ['docker', 'build', '-t', 'churn-prediction-api:latest', '.'],
    capture_output=True, text=True, cwd=os.getcwd()
)

if result.returncode == 0:
    print("Docker image built successfully!")
    print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
else:
    print(f"Build failed!\n{result.stderr}")

In [ ]:
# List Docker images
result = subprocess.run(
    ['docker', 'images', 'churn-prediction-api'],
    capture_output=True, text=True
)
print("Docker Images:")
print(result.stdout)

## Step 8: Run the Docker Container

In [ ]:
# Stop any existing container with the same name
subprocess.run(['docker', 'stop', 'churn-api'], capture_output=True)
subprocess.run(['docker', 'rm', 'churn-api'], capture_output=True)

# Run the container
result = subprocess.run(
    ['docker', 'run', '-d',
     '--name', 'churn-api',
     '-p', '8000:8000',
     '-e', 'SECRET_KEY=mlops-secret-key-2024',
     '-e', 'API_KEYS=mlops-api-key-001,mlops-api-key-002',
     'churn-prediction-api:latest'],
    capture_output=True, text=True
)

if result.returncode == 0:
    container_id = result.stdout.strip()[:12]
    print(f"Container started! ID: {container_id}")
else:
    print(f"Failed to start container: {result.stderr}")

In [ ]:
import time
import requests

# Wait for container to be ready
print("Waiting for container to start...")
time.sleep(5)

# Check container status
result = subprocess.run(
    ['docker', 'ps', '--filter', 'name=churn-api', '--format', '{{.Status}}'],
    capture_output=True, text=True
)
print(f"Container Status: {result.stdout.strip()}")

# Check container logs
result = subprocess.run(
    ['docker', 'logs', 'churn-api', '--tail', '10'],
    capture_output=True, text=True
)
print(f"\nContainer Logs:\n{result.stdout}")

## Step 9: Test the Containerized API

In [ ]:
import json

BASE_URL = "http://localhost:8000"

# Test health endpoint
print("=" * 50)
print("TEST: Health Check")
print("=" * 50)
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5)
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"Error: {e}")

# Test prediction with API Key
print("\n" + "=" * 50)
print("TEST: Prediction with API Key")
print("=" * 50)
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={
            "Gender": "Female", "SeniorCitizen": 1, "Tenure": 2,
            "MonthlyCharges": 95.0, "Contract": "Month-to-month",
            "PaymentMethod": "Electronic check", "TotalCharges": 190.0
        },
        headers={"X-API-Key": "mlops-api-key-001"},
        timeout=5
    )
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")

print("\n✅ Containerized API is working!")

## Step 10: Cleanup

In [ ]:
# Stop and remove container (uncomment to execute)
# subprocess.run(['docker', 'stop', 'churn-api'], capture_output=True)
# subprocess.run(['docker', 'rm', 'churn-api'], capture_output=True)
# print("Container stopped and removed")

print("Files created in this experiment:")
for f in ['Dockerfile', 'docker-compose.yml', 'requirements.txt', '.dockerignore']:
    if os.path.exists(f):
        print(f"  ✅ {f}")

print("\nDocker commands:")
print("  Build:   docker build -t churn-prediction-api .")
print("  Run:     docker run -d -p 8000:8000 --name churn-api churn-prediction-api")
print("  Compose: docker compose up -d")
print("  Logs:    docker logs churn-api")
print("  Stop:    docker stop churn-api && docker rm churn-api")